# Gerar HTML - Consolidação dos plots e tabela, para Laudo de IA de uma OS

Autor:  Viviane da Rosa Sommer

Atualizado: 22/08/2025

Objetivo: Gerar um arquivo HTML consolidando os plots de análise, informações dos modelos de IA utilizados e uma tabela detalhada dos eventos detectados, para compor o laudo automatizado de uma OS.

## Importações necessárias

In [ ]:
import pandas as pd
import glob
import os
import base64
import json
import re

## Declaração de Constantes e Modelos

In [ ]:
operations = [d for d in os.listdir("fase1-0-7-3/validas") if os.path.isdir(os.path.join("fase1-0-7-3/validas", d))]

model_name = 'Classificação (coral_yolov11_class_8c_V4), Segmentação (coral_yolov11_segm_2k_V6.1_parcial) e LLM (gpt-4o)'
model_desc = 'Os modelos foram treinados com imagens de riser, portanto entendem melhor imagens nesse contexto.'

inference_text = "As imagens abaixo foram extraídas da pasta FOTOS da operação e filtradas para análise apenas aquelas que contêm duto, mostrando o fundo do mar, sem o chão visível. Em cada imagem, exibimos: a original, o contorno do coral-sol identificado pelo modelo de segmentação, a classificação (Positiva ou Negativa) de coral-sol com a respectiva probabilidade atribuída pelo modelo (limiar de 0.5). Por fim, a imagem é analisada pelo ChatGPT, que informa a probabilidade de presença de coral-sol, a justificativa e o valor de profundidade apresentado no overlay."
table_text = "Tabela com os eventos encontrados pelo modelo de IA onde a confiança foi superior ou igual à linha de corte."

PATH_LLM_JSONS = "fase3-0-7-3/"
PATH_PLOTS = "fase4-0-7-3/"
PATH_OUTPUT = "fase5-0-7-3/"

## Funções necessárias

In [ ]:
def img_to_base64(path: str) -> str:
    """
    Converts an image file to a base64-encoded data URI string.

    Args:
        path (str): Path to the image file.

    Returns:
        str: Base64-encoded data URI of the image.
    """
    with open(path, "rb") as img_file:
        ext = os.path.splitext(path)[1][1:]
        b64 = base64.b64encode(img_file.read()).decode()
        return f'data:image/{ext};base64,{b64}'


def img_tag(path: str, width: str = "600px") -> str:
    """
    Generates an HTML <img> tag with base64-encoded image and custom styling.

    Args:
        path (str): Path to the image file.
        width (str, optional): Max width of the image. Defaults to "600px".

    Returns:
        str: HTML string for displaying the image.
    """
    return f'<div style="text-align:center;margin:20px 0;"><img src="{img_to_base64(path)}" class="zoomable-img" style="max-width:{width};border-radius:10px;box-shadow:0 2px 8px #888;cursor:pointer;"></div>'


def extract_lda(text: str) -> str:
    """
    Extracts the last floating-point or integer number from a string.

    Args:
        text (str): Input text.

    Returns:
        str: The last number found in the text, or an empty string if none found.
    """
    matches = re.findall(r'(\d+(?:\.\d+)?)', text)
    if matches:
        return matches[-1]
    return ''


def get_llm_texts(llm_dir: str) -> dict:
    """
    Loads LLM response texts from JSON files in a directory.

    Args:
        llm_dir (str): Directory containing JSON files.

    Returns:
        dict: Dictionary mapping file base names to LLM response texts.
    """
    llm_texts = {}
    for json_file in glob.glob(os.path.join(llm_dir, '*.json')):
        base_name = os.path.splitext(os.path.basename(json_file))[0]
        with open(json_file, 'r', encoding='utf-8') as f:
            llm_data = json.load(f)
            text = llm_data.get('prompt_1', {}).get('gpt-4o', {}).get('resposta', '')
            llm_texts[base_name] = text
    return llm_texts


def make_table_rows(imgs: list, llm_texts: dict) -> str:
    """
    Creates HTML table rows with image names, extracted LDA values, and image links.

    Args:
        imgs (list): List of image file paths.
        llm_texts (dict): Dictionary mapping image keys to LLM response texts.

    Returns:
        str: HTML string with sorted table rows.
    """
    rows_data = []
    for img_path in imgs:
        img_name = os.path.basename(img_path)
        img_key = os.path.splitext(img_name)[0]
        lda_val = extract_lda(llm_texts.get(img_key, ''))
        try:
            lda_float = float(lda_val)
        except Exception:
            lda_float = float('inf')
        plot_link = f'<a href="{img_to_base64(img_path)}" class="plot-link">{img_name}</a>'
        rows_data.append((lda_float, f'<tr><td>{img_name}</td><td>{lda_val}</td><td>{plot_link}</td></tr>'))
    rows_data.sort(key=lambda x: x[0])
    return '\n'.join([row for _, row in rows_data])

## Gerar HTML consolidado

In [ ]:
for op in operations:
    plots_dir = PATH_PLOTS + op
    llm_dir = PATH_LLM_JSONS + op
    output_html = PATH_OUTPUT + op +'.html'
    os_number = op.split("_")[-1]
    plots_imgs = sorted(glob.glob(os.path.join(plots_dir, '*')))
    plot_files = {os.path.splitext(os.path.basename(p))[0]: p for p in plots_imgs}
    llm_texts = get_llm_texts(llm_dir)
    plots_html = ''.join([img_tag(img) for img in plots_imgs])
    table_html = f"""
    <div class="section table-section">
    <h2>Tabela de Detalhes das Imagens</h2>
    <table class="styled-table">
    <thead>
    <tr>
    <th>Nome da Imagem</th>
    <th>LDA</th>
    <th>Plots</th>
    </tr>
    </thead>
    <tbody>
    {make_table_rows(plots_imgs, llm_texts)}
    </tbody>
    </table>
    </div>
    """
    css = """
    <style>
    body { font-family: 'Segoe UI', Arial, sans-serif; background: #f8f9fa; margin: 0; padding: 0; }
    h1 { text-align: center; color: #222; margin-top: 30px; }
    h2 { color: #009879; margin: 5px 0 10px 0; text-align: left; }
    .section, .scroll-section, .section.table-section { background: #fff; border-radius: 10px; margin: 30px auto; padding: 25px 35px; max-width: 1200px; box-shadow: 0 2px 12px #ddd; }
    .scroll-section { max-height: 600px; overflow-y: auto; }
    .styled-table { border-collapse: collapse; margin: 25px auto; font-size: 1.1em; min-width: 600px; background-color: #fff; border-radius: 12px 12px 0 0; overflow: hidden; box-shadow: 0 2px 20px #aaa; width: 100%; text-align: center; }
    .styled-table thead tr { background-color: #009879; color: #fff; text-align: center; font-weight: bold; }
    .styled-table th, .styled-table td { padding: 12px 18px; text-align: center; }
    .styled-table tbody tr { border-bottom: 1px solid #dddddd; }
    .styled-table tbody tr:nth-of-type(even) { background-color: #f3f3f3; }
    .styled-table tbody tr:last-of-type { border-bottom: 2px solid #009879; }
    #img-modal { display: none; position: fixed; z-index: 1000; left: 0; top: 0; width: 100vw; height: 100vh; background: rgba(0,0,0,0.85); align-items: center; justify-content: center; }
    #img-modal img { max-width: 90vw; max-height: 90vh; border-radius: 10px; box-shadow: 0 2px 12px #111; }
    #img-modal.show { display: flex; }
    </style>
    """
    js = """
    <script>
    document.addEventListener('DOMContentLoaded', function() {
        var modal = document.getElementById('img-modal');
        var modalImg = document.getElementById('img-modal-img');
        document.querySelectorAll('.zoomable-img').forEach(function(img) {
            img.onclick = function() {
                modalImg.src = this.src;
                modal.classList.add('show');
            }
        });
        modal.onclick = function(e) {
            if (e.target === modal) {
                modal.classList.remove('show');
            }
        }
        document.querySelectorAll('a.plot-link').forEach(function(link) {
            link.onclick = function(e) {
                e.preventDefault();
                modalImg.src = this.href;
                modal.classList.add('show');
            }
        });
    });
    </script>
    <div id="img-modal"><img id="img-modal-img" src=""></div>
    """
    html = f"""
    <html>
    <head>
    <meta charset="utf-8">
    <title>Relatório de análise da operação feito por IA</title>
    {css}
    </head>
    <body>
    <h1>Relatório de análise da operação feito por IA</h1>
    <div class="section">
    <h2>Informações cadastrais da operação</h2>
    <p><b>Número da OS:</b> {os_number}</p>
    </div>
    <div class="section">
    <h2>Informações sobre o modelo de IA utilizado</h2>
    <p><b>Nome do modelo utilizado:</b> {model_name}</p>
    <p><b>Recomendações e limitações do modelo:</b> {model_desc}</p>
    </div>
    <div class="scroll-section">
    <h2>Imagens onde foi identificada a probabilidade de ocorrência</h2>
    <p>{inference_text}</p>
    {plots_html}
    </div>
    {table_html}
    {js}
    </body>
    </html>
    """
    with open(output_html, 'w', encoding='utf-8') as f:
        f.write(html)